# Desafio: filtro de spam com Naive Bayes do zero
Implementação com Python, NumPy e re, sem scikit-learn.
0 = normal; 1 = spam. As colunas de predict_proba seguem modelo.classes_.
Este dataset próprio tem 48 mensagens (24 de cada classe).

## 1. Criar o dataset
No machine learning: estes são os exemplos e suas respostas conhecidas.

In [ ]:
import re
import numpy as np 

#criando 24 mensagens de spam
spam = [
    "Oferta exclusiva de celular com desconto para você",
    "Resgate seu bônus em dinheiro acessando nosso site",
    "Você foi sorteado para receber um prêmio especial",
    "Desconto exclusivo na compra do seu celular hoje",
    "Cadastre seu número e receba dinheiro imediatamente",
    "Últimas vagas para ganhar renda extra sem esforço",
    "Seu cupom premiado libera uma oferta exclusiva",
    "Resgate seu prêmio em dinheiro antes do prazo",
    "Compre dois produtos e receba um bônus especial",
    "Investimento com retorno garantido em poucas horas",
    "Acesse o site e confirme seu cupom de desconto",
    "Você ganhou um celular exclusivo resgate agora",
    "Solicite seu crédito aprovado sem consulta",
    "Uma oferta secreta espera por você no site",
    "Multiplique seu dinheiro com nosso plano exclusivo",
    "Crédito aprovado com bônus exclusivo para você",
    "Participe do sorteio e concorra a dinheiro",
    "Seu prêmio especial precisa ser resgatado agora",
    "Desconto relâmpago em todos os produtos da loja",
    "Oferta de investimento com dinheiro garantido",
    "Cupom exclusivo para comprar celular com desconto",
    "Receba dinheiro extra ao confirmar seu cadastro",
    "Você tem um bônus de compra esperando no site",
    "Resgate agora seu cupom exclusivo e receba um bônus",
]

#criando 24 mensagens normais
normal = [
    "O professor disponibilizou os exercícios na plataforma",
    "Nosso grupo vai estudar na biblioteca depois da aula",
    "O prazo para enviar o trabalho termina na sexta",
    "Vamos estudar os exercícios na biblioteca amanhã",
    "A apresentação do projeto será na próxima semana",
    "Enviei os documentos solicitados pela secretaria",
    "A aula de programação mudou para o laboratório",
    "O professor confirmou a apresentação do trabalho",
    "Podemos revisar o código depois do almoço",
    "Minha irmã vai chegar de viagem no sábado",
    "A biblioteca estará fechada durante o feriado",
    "Enviei o código do projeto para nosso grupo",
    "O resultado da prova já está na plataforma",
    "Preciso devolver o livro que peguei emprestado",
    "O laboratório recebeu novos computadores para as aulas",
    "A secretaria confirmou o prazo dos documentos",
    "Nosso encontro para o almoço ficou para meio dia",
    "O grupo terminou a revisão dos exercícios",
    "A professora pediu ajustes na conclusão do relatório",
    "A aula discutiu como reconhecer uma oferta de prêmio falso",
    "Minha mãe deixou o jantar pronto na geladeira",
    "Precisamos organizar as referências do trabalho",
    "O ônibus da faculdade vai sair mais cedo",
    "A apresentação dos exercícios será depois da aula",
]

textos = spam + normal
y = np.array([1] * len(spam) + [0] * len(normal))
nomes = {0: "normal", 1: "spam"}
print(f"Dataset: {len(textos)} mensagens; {len(spam)} spam e {len(normal)} normais.")

Dataset: 48 mensagens; 24 spam e 24 normais.


## 2. Separar treino e teste
No ML: o treino serve para aprender; o teste serve para avaliar.
Reservamos uma de cada quatro mensagens de cada classe: 36 treino, 12 teste.
A divisão é fixa e mantém as duas classes representadas nos dois conjuntos.

In [ ]:
indices = np.arange(len(textos))
indices_treino = indices[indices % 4 != 3]
indices_teste = indices[indices % 4 == 3]
textos_treino = [textos[i] for i in indices_treino]
textos_teste = [textos[i] for i in indices_teste]
y_treino = y[indices_treino]
y_teste = y[indices_teste]
print("Treino:", len(textos_treino), "| Teste:", len(textos_teste))
print("Treino por classe:", {nomes[c]: int(np.sum(y_treino == c)) for c in [0, 1]})
print("Teste por classe:", {nomes[c]: int(np.sum(y_teste == c)) for c in [0, 1]})

Treino: 36 | Teste: 12
Treino por classe: {'normal': 18, 'spam': 18}
Teste por classe: {'normal': 6, 'spam': 6}


## 3. Criar o vocabulário apenas com o treino
No ML: pré-processamento. Usar o teste para criar o vocabulário seria vazamento.
Tokenizar significa colocar em minúsculas e separar as palavras sem pontuação.

In [ ]:
def tokenizar(texto):
    return re.findall(r"[a-záàâãéêíóôõúüç]+", texto.lower())


def construir_vocabulario(textos_treino):
    palavras = sorted({p for texto in textos_treino for p in tokenizar(texto)})
    return {palavra: i for i, palavra in enumerate(palavras)}


vocab = construir_vocabulario(textos_treino)
print("Palavras no vocabulário de treino:", len(vocab))

Palavras no vocabulário de treino: 163


## 4. Transformar textos em vetores de contagem
No ML: construção da matriz X, com uma linha por mensagem e uma coluna por palavra.
Repetições são contadas. Palavras fora do vocabulário de treino são ignoradas.
Treino, teste e mensagens novas usam exatamente as mesmas colunas.

In [ ]:
def vetorizar(textos, vocab):
    X = np.zeros((len(textos), len(vocab)), dtype=int)
    for i, texto in enumerate(textos):
        for palavra in tokenizar(texto):
            if palavra in vocab:
                X[i, vocab[palavra]] += 1
    return X


X_treino = vetorizar(textos_treino, vocab)
X_teste = vetorizar(textos_teste, vocab)
print("X_treino:", X_treino.shape, "| X_teste:", X_teste.shape)

X_treino: (36, 163) | X_teste: (12, 163)


## 5 a 11. Implementar e treinar o modelo
No ML: fit aprende os parâmetros; predict e predict_proba fazem a inferência.

5: prior = número de mensagens da classe / número total de mensagens. <br>
6: somar as ocorrências de cada palavra dentro de cada classe. <br>
7: P(palavra|classe) = (contagem + alpha) / (total de tokens + alpha * tamanho do vocabulário). <br>
8: usar logaritmos para evitar underflow ao combinar muitas probabilidades. <br>
9: reunir esses cálculos em fit(). <br>
10: escolher a classe com maior log-pontuação em predict(). <br>
11: em predict_proba(), subtrair o máximo, exponenciar e normalizar cada linha. <br>

O modelo recebe contagens, não textos. A tokenização fica fora da classe. 

In [ ]:
class NaiveBayesDoZero:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        assert self.alpha > 0, "alpha deve ser positivo"
        assert np.all(X >= 0), "as contagens não podem ser negativas"
        self.classes_ = np.unique(y)

        #calculando os priors das classes
        self.prior_ = np.array([np.mean(y == c) for c in self.classes_])

        #contando palavras e o total de tokens por classe
        self.contagens_ = np.array([X[y == c].sum(axis=0) for c in self.classes_])
        self.totais_ = self.contagens_.sum(axis=1, keepdims=True)

        #aplicando a suavização de laplace
        self.prob_palavras_ = (
            (self.contagens_ + self.alpha)
            / (self.totais_ + self.alpha * X.shape[1])
        )

        #guardando as probabilidades em log
        self.log_prior_ = np.log(self.prior_)
        self.log_prob_ = np.log(self.prob_palavras_)
        return self

    def log_pontuacoes(self, X):
        #somando log prior e as contribuições das palavras
        return self.log_prior_ + X @ self.log_prob_.T

    def predict(self, X):
        pontuacoes = self.log_pontuacoes(X)
        return self.classes_[np.argmax(pontuacoes, axis=1)]

    def predict_proba(self, X):
        pontuacoes = self.log_pontuacoes(X)
        pontuacoes = pontuacoes - pontuacoes.max(axis=1, keepdims=True)
        probabilidades = np.exp(pontuacoes)
        return probabilidades / probabilidades.sum(axis=1, keepdims=True)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


modelo = NaiveBayesDoZero(alpha=1.0)
modelo.fit(X_treino, y_treino)
print("Ordem das classes:", modelo.classes_)
print("Priors:", modelo.prior_)
print("Total de tokens por classe:", modelo.totais_.ravel())

Ordem das classes: [0 1]
Priors: [0.5 0.5]
Total de tokens por classe: [138 136]


## 12. Avaliar: acurácia e matriz de confusão
No ML: avaliação em dados que não participaram do treinamento.
Na matriz, linhas são as classes reais e colunas são as classes previstas.
A ordem é [normal, spam]. A diagonal contém os acertos.

In [ ]:
y_previsto = modelo.predict(X_teste)
prob_teste = modelo.predict_proba(X_teste)
acuracia = modelo.score(X_teste, y_teste)

matriz = np.zeros((len(modelo.classes_), len(modelo.classes_)), dtype=int)
for i, classe_real in enumerate(modelo.classes_):
    for j, classe_prevista in enumerate(modelo.classes_):
        matriz[i, j] = np.sum((y_teste == classe_real) & (y_previsto == classe_prevista))

print(f"Acurácia: {acuracia:.2%} ({np.sum(y_teste == y_previsto)}/{len(y_teste)})")
print("Matriz de confusão: linhas = real; colunas = previsto; ordem = normal, spam")
print(matriz)
for texto, real, previsto in zip(textos_teste, y_teste, y_previsto):
    print(f"Real: {nomes[real]:6} | Previsto: {nomes[previsto]:6} | {texto}")

Acurácia: 91.67% (11/12)
Matriz de confusão: linhas = real; colunas = previsto; ordem = normal, spam
[[5 1]
 [0 6]]
Real: spam   | Previsto: spam   | Desconto exclusivo na compra do seu celular hoje
Real: spam   | Previsto: spam   | Resgate seu prêmio em dinheiro antes do prazo
Real: spam   | Previsto: spam   | Você ganhou um celular exclusivo resgate agora
Real: spam   | Previsto: spam   | Crédito aprovado com bônus exclusivo para você
Real: spam   | Previsto: spam   | Oferta de investimento com dinheiro garantido
Real: spam   | Previsto: spam   | Resgate agora seu cupom exclusivo e receba um bônus
Real: normal | Previsto: normal | Vamos estudar os exercícios na biblioteca amanhã
Real: normal | Previsto: normal | O professor confirmou a apresentação do trabalho
Real: normal | Previsto: normal | Enviei o código do projeto para nosso grupo
Real: normal | Previsto: normal | A secretaria confirmou o prazo dos documentos
Real: normal | Previsto: spam   | A aula discutiu como reconhecer uma

## 13. Testar mensagens novas
No ML: inferência. Reutilizamos o vocabulário e o modelo já treinados.

In [ ]:
mensagens_novas = [
    "Resgate seu bônus exclusivo em dinheiro",
    "O professor enviou os exercícios para o grupo",
    "galáxia interplanetária",
]
X_novas = vetorizar(mensagens_novas, vocab)
previsoes_novas = modelo.predict(X_novas)
prob_novas = modelo.predict_proba(X_novas)
for texto, previsto, probabilidades in zip(mensagens_novas, previsoes_novas, prob_novas):
    probs = {nomes[c]: round(float(p), 4) for c, p in zip(modelo.classes_, probabilidades)}
    print(f"{texto!r} -> {nomes[previsto]} | {probs}")
print("Se nenhuma palavra for conhecida, as probabilidades serão os priors.")
print("No empate, np.argmax escolhe a primeira classe: neste modelo, normal.")

'Resgate seu bônus exclusivo em dinheiro' -> spam | {'normal': 0.0002, 'spam': 0.9998}
'O professor enviou os exercícios para o grupo' -> normal | {'normal': 0.9993, 'spam': 0.0007}
'galáxia interplanetária' -> normal | {'normal': 0.5, 'spam': 0.5}
Se nenhuma palavra for conhecida, as probabilidades serão os priors.
No empate, np.argmax escolhe a primeira classe: neste modelo, normal.


## 14. Explicar duas previsões
No ML: interpretação do modelo. Mostramos a contribuição real de cada palavra.
log-pontuação = log prior + soma de contagem * log P(palavra|classe).
Palavras desconhecidas não contribuem para a pontuação.

In [ ]:
def explicar_previsao(texto, modelo, vocab):
    X = vetorizar([texto], vocab)
    pontuacoes = modelo.log_pontuacoes(X)[0]
    probabilidades = modelo.predict_proba(X)[0]
    print(f"\nMensagem: {texto}")
    print(f"{'palavra':16} {'contagem':>8} {'contrib. normal':>16} {'contrib. spam':>14}")
    for palavra, j in vocab.items():
        if X[0, j] > 0:
            contribuicoes = X[0, j] * modelo.log_prob_[:, j]
            print(f"{palavra:16} {X[0, j]:8d} {contribuicoes[0]:16.4f} {contribuicoes[1]:14.4f}")
    desconhecidas = sorted(set(tokenizar(texto)) - set(vocab))
    print("Palavras ignoradas:", desconhecidas)
    print("Log priors [normal, spam]:", np.round(modelo.log_prior_, 4))
    print("Log-pontuações finais:", np.round(pontuacoes, 4))
    print("Probabilidades:", np.round(probabilidades, 4))
    print("Previsão:", nomes[modelo.predict(X)[0]])
    print("A classe escolhida tem a maior log-pontuação; menos negativo é maior.")


explicar_previsao(mensagens_novas[0], modelo, vocab)
explicar_previsao(mensagens_novas[1], modelo, vocab)


Mensagem: Resgate seu bônus exclusivo em dinheiro
palavra          contagem  contrib. normal  contrib. spam
bônus                   1          -5.7071        -4.3141
dinheiro                1          -5.7071        -3.9087
em                      1          -5.7071        -4.3141
exclusivo               1          -5.7071        -4.6018
resgate                 1          -5.7071        -5.0073
seu                     1          -5.7071        -3.5032
Palavras ignoradas: []
Log priors [normal, spam]: [-0.6931 -0.6931]
Log-pontuações finais: [-34.9358 -26.3425]
Probabilidades: [2.000e-04 9.998e-01]
Previsão: spam
A classe escolhida tem a maior log-pontuação; menos negativo é maior.

Mensagem: O professor enviou os exercícios para o grupo
palavra          contagem  contrib. normal  contrib. spam
exercícios              1          -4.6085        -5.7004
grupo                   1          -4.6085        -5.7004
o                       2          -6.1361       -10.0146
os                  

## Parágrafo sobre os erros do modelo

No teste com 12 mensagens, o modelo acertou 11 e errou 1, com acurácia de 91,67%. Uma mensagem normal foi classificada como spam, e todos os spams foram classificados corretamente. O modelo usa contagens e não entende o contexto nem a ordem das palavras; por isso, uma mensagem legítima que discuta ofertas ou prêmios pode receber uma pontuação alta para spam. Palavras fora do vocabulário são ignoradas. Como o dataset é pequeno, criado para estudo e avaliado com uma única divisão, esse resultado não demonstra o desempenho em mensagens reais de outros contextos.

O erro foi a mensagem **“A aula discutiu como reconhecer uma oferta de prêmio falso”**. Ela é uma mensagem normal sobre uma aula, mas contém palavras associadas a spam no treino, como “oferta” e “prêmio”. A representação por contagens não interpreta que se trata de uma discussão sobre o assunto.

Nas duas previsões explicadas, “Resgate seu bônus exclusivo em dinheiro” foi classificada como spam porque a soma das contribuições dessas palavras e do log prior foi maior para spam. “O professor enviou os exercícios para o grupo” foi classificada como normal porque a soma foi maior para normal; “enviou” foi ignorada porque não aparece no vocabulário do treino.